# Forklift Movie Preprocess

元動画から front / rear 動画を切り出し、
同じトリム区間の音声 wav も `data/movie_preprocess/<category>/<environment>/` に保存します。
黒帯除去、frame_cut、無地フレームtrimなど既存の前処理が終わった後に、front / rear それぞれの差分ピークからviewごとの基準フレーム間隔を推定し、その等間隔グリッドで代表フレームを残します。
`data/frame_cut.txt` に指定がある動画は、その後に前後フレームをカットしてから無地フレームをトリムします。`#` で始まる行はコメントとして読み飛ばし、`all` 行がある場合は全動画共通のカット数として個別動画分に加算します。4列目を指定した個別動画だけ、内容更新フレーム間引きの差分閾値を上書きします。


このノートブックは、元動画を学習・推論で扱いやすい `front.mp4` / `rear.mp4` / `audio.wav` にそろえる前処理です。黒帯除去、前後カメラ分割、trim、音声抽出、manifest 保存までを同じルールで行います。


## 処理の流れ

1. 元動画をframe_cut、黒帯除去、無地フレームtrimの順に前処理します。
2. front/rearそれぞれの差分ピークからviewごとの基準フレーム間隔を推定し、等間隔グリッドで代表フレームを選びます。
3. front/rear動画、音声、manifestを `data/movie_preprocess` に保存します。


## 0. パッケージ


In [176]:
# ------------------------------------------------------------
# セル概要: パッケージ確認
# ------------------------------------------------------------
# - 動画前処理に必要な OpenCV / numpy / pandas / matplotlib などを確認します。
# - ffmpeg / ffprobe はPythonパッケージではなく実行ファイルなので、後続セルで存在確認します。
# ------------------------------------------------------------

# %pip install jupyterlab==4.4.1 matplotlib==3.10.1 numpy==2.2.4 pandas==2.2.3 opencv-python-headless==4.11.0.86


In [177]:
# ------------------------------------------------------------
# セル概要: ライブラリ読み込み
# ------------------------------------------------------------
# - 動画読み書き、音声抽出、manifest 作成、ファイル削除に使う標準ライブラリと OpenCV を読み込みます。
# - 以降のセルでは前処理済み動画と音声を同じルールで出力します。
# ------------------------------------------------------------

from __future__ import annotations

from pathlib import Path
from typing import Any

import importlib
import shutil
import subprocess
import sys

import cv2
import numpy as np
import pandas as pd
from IPython.display import display


def _find_import_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "forklift_features").exists():
            return candidate
    raise FileNotFoundError(f"Could not find src/forklift_features from {start}")


_IMPORT_ROOT = _find_import_root()
src_path = str(_IMPORT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from forklift_features import content_update
content_update = importlib.reload(content_update)


In [ ]:
# ------------------------------------------------------------
# セル概要: 前処理設定
# ------------------------------------------------------------
# - 入力動画の場所、出力先、一時的な画面キャプチャfps復元、内容更新フレーム間引き、黒帯除去、音声サンプリング周波数をまとめます。
# - ここを変えると全動画の出力仕様が変わるため、manifest にも設定値を記録します。
# ------------------------------------------------------------

DATA_DIR = Path("../data")
ORI_DATA_DIR = DATA_DIR / "ori"
INPUT_VIDEO_DIRS = {
    "normal": ORI_DATA_DIR / "normal",
    "anomaly": ORI_DATA_DIR / "anomaly",
}
OUTDOOR_LIST_PATH = DATA_DIR / "outdoor_list.txt"
FRAME_CUT_PATH = DATA_DIR / "frame_cut.txt"
MOVIE_PREPROCESS_OUTPUT_DIR = DATA_DIR / "movie_preprocess"
MOVIE_PREPROCESS_MANIFEST_PATH = MOVIE_PREPROCESS_OUTPUT_DIR / "movie_preprocess_manifest.csv"

VIDEO_PROCESSING_ENABLED = True
# 空なら全動画を処理します。例: 'sample_001' / 'sample_001.mp4' / ['sample_001', 'sample_002.mp4']
MOVIE_PREPROCESS_TARGET_VIDEO_NAMES: str | list[str] | tuple[str, ...] | set[str] | None = None
# MOVIE_PREPROCESS_TARGET_VIDEO_NAMES = [
#     # "1079",
#     # "1001",
#     # "002_他パレットに衝突",
#     # "032_停車中に車両に衝突される",
#     "1041"
# ]
MOVIE_PREPROCESS_OVERWRITE = False
MOVIE_PREPROCESS_DELETE_STALE = True
MOVIE_PREPROCESS_FOURCC = "mp4v"
MOVIE_PREPROCESS_MIN_OUTPUT_FPS = 1.0
# 既存前処理後に、front/rear別で内容更新フレームだけを残します。
CONTENT_UPDATE_FRAME_THINNING_ENABLED = True
DUPLICATE_DIFF_THRESHOLD = content_update.DEFAULT_CONTENT_UPDATE_DIFF_THRESHOLD
FLOW_TARGET_DT = content_update.DEFAULT_FLOW_TARGET_DT
# front/rearそれぞれの差分ピークから基準フレーム間隔を推定し、view別の等間隔グリッドで代表フレームを残します。
CONTENT_UPDATE_GRID_PEAK_SCORE_PERCENTILE = 60.0
CONTENT_UPDATE_GRID_BASE_INTERVAL_WINDOW_UPDATES = 5
CONTENT_UPDATE_GRID_BASE_INTERVAL_PERCENTILE = 25.0
CONTENT_UPDATE_GRID_FALLBACK_INTERVAL_SEC = FLOW_TARGET_DT
CONTENT_UPDATE_GRID_INCLUDE_LAST_FRAME = False
STATIC_SCENE_MIN_SEC = content_update.DEFAULT_STATIC_SCENE_MIN_SEC
FLOW_NORMALIZE_BY_DT = content_update.DEFAULT_FLOW_NORMALIZE_BY_DT
CONTENT_UPDATE_FRAME_THINNING_RESIZE_WIDTH = content_update.DEFAULT_CONTENT_UPDATE_RESIZE_WIDTH
CONTENT_UPDATE_FRAME_THINNING_GAUSSIAN_BLUR_KERNEL: int | None = content_update.DEFAULT_CONTENT_UPDATE_GAUSSIAN_BLUR_KERNEL
CONTENT_UPDATE_TILE_GRID_ROWS = content_update.DEFAULT_CONTENT_UPDATE_TILE_GRID_ROWS
CONTENT_UPDATE_TILE_GRID_COLS = content_update.DEFAULT_CONTENT_UPDATE_TILE_GRID_COLS
CONTENT_UPDATE_TILE_TOP_FRACTION = content_update.DEFAULT_CONTENT_UPDATE_TILE_TOP_FRACTION
CONTENT_UPDATE_TILE_ACTIVE_THRESHOLD_RATIO = content_update.DEFAULT_CONTENT_UPDATE_TILE_ACTIVE_THRESHOLD_RATIO
CONTENT_UPDATE_TILE_MIN_ACTIVE_TILES = content_update.DEFAULT_CONTENT_UPDATE_TILE_MIN_ACTIVE_TILES
CONTENT_UPDATE_TILE_MIN_ACTIVE_RATIO = content_update.DEFAULT_CONTENT_UPDATE_TILE_MIN_ACTIVE_RATIO
VIDEO_SAMPLE_FPS = 2
VIDEO_MAX_SAMPLE_FRAMES = 200
VIDEO_LETTERBOX_BLACK_THRESHOLD = 8
VIDEO_UNIFORM_DOMINANT_RATIO_THRESHOLD = 0.95
VIDEO_UNIFORM_COLOR_TOLERANCE = 30.0
AUDIO_SAMPLE_RATE = 22050
AUDIO_MONO_CHANNELS = 1

FFMPEG_PATH = shutil.which("ffmpeg")
FFPROBE_PATH = shutil.which("ffprobe")
if FFMPEG_PATH is None:
    raise FileNotFoundError("ffmpeg is required but was not found in PATH.")
if FFPROBE_PATH is None:
    raise FileNotFoundError("ffprobe is required but was not found in PATH.")

## 1. ユーティリティ


In [179]:
# ------------------------------------------------------------
# セル概要: 前処理ユーティリティ
# ------------------------------------------------------------
# - 動画メタデータ取得、黒帯検出、front/rear 分割、重複フレーム除去、古い成果物掃除などの補助関数群です。
# - 単体関数を小さく分け、バッチ処理本体では「何をするか」が読めるようにしています。
# ------------------------------------------------------------

# 関数メモ: ensure_parent_directory
# - 出力ファイルを書き込む前に親ディレクトリを作成します。
# - 関数本体のあちこちで `mkdir` を重複させないための小さな共通処理です。

def ensure_parent_directory(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


# 関数メモ: read_video_metadata
# - OpenCV で動画の幅・高さ・FPS・フレーム数・長さを取得します。
# - 前処理の可否判定、manifest 記録、trim 範囲の計算に使います。

def read_video_metadata(video_path: Path) -> dict[str, Any]:
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Failed to open video: {video_path}")

    metadata = {
        "video_path": str(video_path),
        "frame_count": int(capture.get(cv2.CAP_PROP_FRAME_COUNT)),
        "fps": float(capture.get(cv2.CAP_PROP_FPS)),
        "width": int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    metadata["duration_sec"] = (
        metadata["frame_count"] / metadata["fps"] if metadata["fps"] > 0 else np.nan
    )
    capture.release()
    return metadata


# 関数メモ: detect_side_letterbox_bounds
# - 左右に入った黒帯・単色帯の境界を、列ごとの暗さや色の均一さから推定します。
# - 上下2画面結合動画を分割する前に、不要な左右余白を落とすために使います。

def detect_side_letterbox_bounds(
    stacked_frame: np.ndarray,
    black_threshold: int = VIDEO_LETTERBOX_BLACK_THRESHOLD,
    min_non_black_ratio: float = 0.01,
) -> tuple[int, int, dict[str, int]]:
    gray = cv2.cvtColor(stacked_frame, cv2.COLOR_RGB2GRAY)
    non_black_ratio = (gray > black_threshold).mean(axis=0)
    valid_columns = np.where(non_black_ratio > min_non_black_ratio)[0]

    if valid_columns.size == 0:
        crop_left = 0
        crop_right = stacked_frame.shape[1]
    else:
        crop_left = int(valid_columns[0])
        crop_right = int(valid_columns[-1]) + 1

    if crop_right <= crop_left:
        crop_left = 0
        crop_right = stacked_frame.shape[1]

    return crop_left, crop_right, {
        "original_width": int(stacked_frame.shape[1]),
        "crop_left": int(crop_left),
        "crop_right": int(crop_right),
        "cropped_width": int(crop_right - crop_left),
        "removed_left_px": int(crop_left),
        "removed_right_px": int(stacked_frame.shape[1] - crop_right),
    }


# 関数メモ: apply_side_crop
# - 推定した左端・右端の crop 位置でフレームを切り出します。
# - crop 範囲外の値はフレーム幅内に補正し、異常な設定で処理が壊れにくいようにします。

def apply_side_crop(stacked_frame: np.ndarray, crop_left: int, crop_right: int) -> np.ndarray:
    frame_width = int(stacked_frame.shape[1])
    left = max(0, min(int(crop_left), frame_width - 1))
    right = max(left + 1, min(int(crop_right), frame_width))
    return stacked_frame[:, left:right, :]


# 関数メモ: choose_common_side_letterbox_crop
# - 複数サンプルフレームから共通して安全に使える左右 crop 幅を決めます。
# - フレームごとに crop が揺れると出力動画が不安定になるため、動画単位の共通値にします。

def choose_common_side_letterbox_crop(stacked_frames: list[np.ndarray]) -> dict[str, int]:
    if not stacked_frames:
        return {
            "frame_count": 0,
            "original_width": 0,
            "crop_left": 0,
            "crop_right": 0,
            "cropped_width": 0,
            "removed_left_px": 0,
            "removed_right_px": 0,
        }

    crop_lefts: list[int] = []
    crop_rights: list[int] = []
    original_width = int(stacked_frames[0].shape[1])
    for stacked_frame in stacked_frames:
        crop_left, crop_right, _ = detect_side_letterbox_bounds(stacked_frame)
        crop_lefts.append(crop_left)
        crop_rights.append(crop_right)

    crop_left = int(np.median(crop_lefts))
    crop_right = int(np.median(crop_rights))
    crop_left = max(0, min(crop_left, original_width - 1))
    crop_right = max(crop_left + 1, min(crop_right, original_width))

    return {
        "frame_count": int(len(stacked_frames)),
        "original_width": original_width,
        "crop_left": int(crop_left),
        "crop_right": int(crop_right),
        "cropped_width": int(crop_right - crop_left),
        "removed_left_px": int(crop_left),
        "removed_right_px": int(original_width - crop_right),
    }


# 関数メモ: preprocess_stacked_frames_remove_letterbox
# - サンプルフレーム群から左右黒帯を推定し、crop 情報をまとめます。
# - 以降の実フレーム処理ではこの crop 情報を使って一貫した切り出しを行います。

def preprocess_stacked_frames_remove_letterbox(
    stacked_frames: list[np.ndarray],
) -> tuple[list[np.ndarray], dict[str, int]]:
    if not stacked_frames:
        return [], choose_common_side_letterbox_crop(stacked_frames)

    crop_info = choose_common_side_letterbox_crop(stacked_frames)
    cropped_frames = [
        apply_side_crop(stacked_frame, crop_info["crop_left"], crop_info["crop_right"])
        for stacked_frame in stacked_frames
    ]
    return cropped_frames, crop_info


# 関数メモ: split_front_rear_frame
# - 上下に積まれた1枚のフレームを front と rear の2枚に分割します。
# - 前処理済み出力を `*_front.mp4` / `*_rear.mp4` に分ける中心処理です。

def split_front_rear_frame(stacked_frame: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    height = stacked_frame.shape[0]
    midpoint = height // 2
    return stacked_frame[:midpoint, :, :], stacked_frame[midpoint:, :, :]


# 関数メモ: is_nearly_uniform_frame
# - 黒画面や単色に近いフレームかどうかを判定します。
# - 冒頭・末尾の無効フレーム trim や、画面がほぼ情報を持たない区間の除外に使います。

def is_nearly_uniform_frame(
    frame: np.ndarray,
    dominant_ratio_threshold: float = VIDEO_UNIFORM_DOMINANT_RATIO_THRESHOLD,
    color_tolerance: float = VIDEO_UNIFORM_COLOR_TOLERANCE,
) -> bool:
    pixels = frame.reshape(-1, 3).astype(np.float32)
    representative_color = np.median(pixels, axis=0)
    pixel_deviation = np.abs(pixels - representative_color).max(axis=1)
    near_representative_ratio = float((pixel_deviation <= color_tolerance).mean())
    return near_representative_ratio >= dominant_ratio_threshold


# 関数メモ: create_mp4_writer
# - OpenCV の VideoWriter を作成し、指定サイズ・FPSのMP4を書ける状態にします。
# - writer が開けない場合は即座に例外にして、空の出力が残る問題を避けます。

def create_mp4_writer(output_path: Path, fps: float, frame_shape: tuple[int, ...]) -> cv2.VideoWriter:
    ensure_parent_directory(output_path)
    height, width = frame_shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*MOVIE_PREPROCESS_FOURCC)
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (int(width), int(height)))
    if not writer.isOpened():
        raise RuntimeError(f"Failed to open video writer: {output_path}")
    return writer


# 関数メモ: iter_source_frames_by_index
# - 元動画フレーム番号リストに従って、必要なフレームだけを順番に読みます。
# - frame_cut / trim 後のフレーム番号リストを後続処理で共通して読み出すために使います。

def iter_source_frames_by_index(
    video_path: Path,
    source_frame_indices: list[int],
):
    target_source_frame_indices = {int(index) for index in source_frame_indices if int(index) >= 0}
    if not target_source_frame_indices:
        return

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Failed to open video: {video_path}")

    last_source_frame_index = max(target_source_frame_indices)
    source_frame_index = 0
    try:
        while capture.isOpened():
            success, frame_bgr = capture.read()
            if not success:
                break
            if source_frame_index > last_source_frame_index:
                break
            if source_frame_index in target_source_frame_indices:
                yield source_frame_index, frame_bgr
            source_frame_index += 1
    finally:
        capture.release()


# 関数メモ: iter_sampled_source_frames
# - 実処理対象のフレーム番号リストから、指定fps相当のサンプルフレームを取り出します。
# - 入力fpsの時間軸でサンプリング間隔を決めるため、frame_cut 後の黒帯推定にも使えます。

def iter_sampled_source_frames(
    video_path: Path,
    source_frame_indices: list[int],
    effective_fps: float,
    sampling_fps: int = VIDEO_SAMPLE_FPS,
    max_frames: int | None = VIDEO_MAX_SAMPLE_FRAMES,
) -> list[np.ndarray]:
    if not source_frame_indices:
        return []

    step = max(int(round(effective_fps / max(sampling_fps, 1))), 1) if effective_fps > 0 else 1
    sampled_source_frame_indices = list(source_frame_indices[::step])
    if max_frames is not None:
        sampled_source_frame_indices = sampled_source_frame_indices[:max_frames]

    return [
        cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        for _, frame_bgr in iter_source_frames_by_index(video_path, sampled_source_frame_indices)
    ]


# 関数メモ: estimate_source_frame_indices_uniform_trim
# - 実処理対象のフレーム番号リストに対して、前後の単色に近い区間を trim する範囲を推定します。
# - frame_cut 前の一時fps復元で間引いた後の時間軸に合わせて trim 量を数えます。

def estimate_source_frame_indices_uniform_trim(
    video_path: Path,
    crop_info: dict[str, int],
    source_frame_indices: list[int],
) -> dict[str, int]:
    total_frame_pairs = 0
    front_uniform_start = 0
    rear_uniform_start = 0
    front_uniform_end = 0
    rear_uniform_end = 0
    front_start_open = True
    rear_start_open = True

    for _, stacked_frame_bgr in iter_source_frames_by_index(video_path, source_frame_indices):
        cropped_frame_bgr = apply_side_crop(
            stacked_frame_bgr,
            crop_info["crop_left"],
            crop_info["crop_right"],
        )
        front_frame_bgr, rear_frame_bgr = split_front_rear_frame(cropped_frame_bgr)
        front_is_uniform = is_nearly_uniform_frame(front_frame_bgr)
        rear_is_uniform = is_nearly_uniform_frame(rear_frame_bgr)

        if front_start_open and front_is_uniform:
            front_uniform_start += 1
        else:
            front_start_open = False

        if rear_start_open and rear_is_uniform:
            rear_uniform_start += 1
        else:
            rear_start_open = False

        front_uniform_end = front_uniform_end + 1 if front_is_uniform else 0
        rear_uniform_end = rear_uniform_end + 1 if rear_is_uniform else 0
        total_frame_pairs += 1

    removed_from_start = max(front_uniform_start, rear_uniform_start)
    removed_from_end = max(front_uniform_end, rear_uniform_end)
    remaining_frame_pairs = max(total_frame_pairs - removed_from_start - removed_from_end, 0)

    return {
        "input_frame_pairs": int(total_frame_pairs),
        "front_uniform_frames_at_start": int(front_uniform_start),
        "rear_uniform_frames_at_start": int(rear_uniform_start),
        "front_uniform_frames_at_end": int(front_uniform_end),
        "rear_uniform_frames_at_end": int(rear_uniform_end),
        "removed_from_start": int(removed_from_start),
        "removed_from_end": int(removed_from_end),
        "remaining_frame_pairs": int(remaining_frame_pairs),
    }


# 関数メモ: select_content_update_source_frame_indices_by_view
# - front/rear別のタイル差分ピークからviewごとの基準フレーム間隔を推定します。
# - 最終的な保持フレームは、しきい値判定そのものではなく等間隔グリッドで選びます。
# - 局所的な小さい動きでも上位タイル差分で拾い、推定またはfallback間隔でフレームが残るようにします。

def select_content_update_source_frame_indices_by_view(
    video_path: Path,
    source_frame_indices: list[int],
    crop_info: dict[str, int],
    effective_fps: float,
    source_duration_sec: float,
    enabled: bool = CONTENT_UPDATE_FRAME_THINNING_ENABLED,
    diff_threshold: float = DUPLICATE_DIFF_THRESHOLD,
) -> tuple[dict[str, list[int]], dict[str, Any]]:
    input_frame_pairs = int(len(source_frame_indices))
    fallback_fps = float(effective_fps) if effective_fps > 0 else float(MOVIE_PREPROCESS_MIN_OUTPUT_FPS)
    peak_score_percentile = float(np.clip(CONTENT_UPDATE_GRID_PEAK_SCORE_PERCENTILE, 0.0, 100.0))
    base_interval_window_updates = max(2, int(CONTENT_UPDATE_GRID_BASE_INTERVAL_WINDOW_UPDATES))
    base_interval_percentile = float(np.clip(CONTENT_UPDATE_GRID_BASE_INTERVAL_PERCENTILE, 0.0, 100.0))
    fallback_interval_sec = max(0.0, float(CONTENT_UPDATE_GRID_FALLBACK_INTERVAL_SEC))
    fallback_interval_frames = max(1, int(round(fallback_interval_sec * fallback_fps))) if fallback_fps > 0 else 1
    summary_kwargs = {
        "source_duration_sec": source_duration_sec,
        "fallback_fps": fallback_fps,
        "static_scene_min_sec": STATIC_SCENE_MIN_SEC,
        "min_output_fps": MOVIE_PREPROCESS_MIN_OUTPUT_FPS,
    }

    def base_info() -> dict[str, Any]:
        return {
            "content_update_frame_thinning_enabled": bool(enabled),
            "content_update_duplicate_diff_threshold": float(diff_threshold),
            "content_update_resize_width": -1 if CONTENT_UPDATE_FRAME_THINNING_RESIZE_WIDTH is None else int(CONTENT_UPDATE_FRAME_THINNING_RESIZE_WIDTH),
            "content_update_static_scene_min_sec": float(STATIC_SCENE_MIN_SEC),
            "content_update_tile_grid_rows": int(CONTENT_UPDATE_TILE_GRID_ROWS),
            "content_update_tile_grid_cols": int(CONTENT_UPDATE_TILE_GRID_COLS),
            "content_update_tile_top_fraction": float(CONTENT_UPDATE_TILE_TOP_FRACTION),
            "content_update_tile_active_threshold_ratio": float(CONTENT_UPDATE_TILE_ACTIVE_THRESHOLD_RATIO),
            "content_update_tile_min_active_tiles": int(CONTENT_UPDATE_TILE_MIN_ACTIVE_TILES),
            "content_update_tile_min_active_ratio": float(CONTENT_UPDATE_TILE_MIN_ACTIVE_RATIO),
            "content_update_flow_target_dt": float(FLOW_TARGET_DT),
            "content_update_grid_peak_score_percentile": float(peak_score_percentile),
            "content_update_grid_base_interval_window_updates": int(base_interval_window_updates),
            "content_update_grid_base_interval_percentile": float(base_interval_percentile),
            "content_update_grid_fallback_interval_sec": float(fallback_interval_sec),
            "content_update_grid_include_last_frame": bool(CONTENT_UPDATE_GRID_INCLUDE_LAST_FRAME),
            "content_update_flow_normalize_by_dt": bool(FLOW_NORMALIZE_BY_DT),
        }

    if not source_frame_indices or not enabled:
        kept_by_view = {"front": list(source_frame_indices), "rear": list(source_frame_indices)}
        info = base_info()
        for view, kept_indices in kept_by_view.items():
            info.update(content_update.summarize_content_update_view(
                f"{view}_content_update",
                enabled=enabled,
                input_frame_pairs=input_frame_pairs,
                kept_indices=kept_indices,
                duplicate_count=0,
                diff_scores=[],
                max_duplicate_run=0,
                max_time_since_content_update=0.0,
                **summary_kwargs,
            ))
            info[f"{view}_content_update_peak_frame_pairs"] = 0
            info[f"{view}_content_update_grid_frame_pairs"] = int(len(kept_indices))
            info[f"{view}_content_update_peak_score_threshold"] = np.nan
            info[f"{view}_content_update_grid_base_interval_frames"] = np.nan
            info[f"{view}_content_update_grid_base_interval_sec"] = np.nan
            info[f"{view}_content_update_grid_interval_frames"] = np.nan
            info[f"{view}_content_update_grid_interval_sec"] = np.nan
        return kept_by_view, info

    states = {
        "front": {"previous_gray": None, "diff_by_ordinal": np.full(input_frame_pairs, np.nan, dtype=float)},
        "rear": {"previous_gray": None, "diff_by_ordinal": np.full(input_frame_pairs, np.nan, dtype=float)},
    }
    ordinal_by_source_frame_index = {int(frame_index): ordinal for ordinal, frame_index in enumerate(source_frame_indices)}

    for source_frame_index, stacked_frame_bgr in iter_source_frames_by_index(video_path, source_frame_indices):
        ordinal = int(ordinal_by_source_frame_index.get(int(source_frame_index), 0))
        cropped_frame_bgr = apply_side_crop(stacked_frame_bgr, crop_info["crop_left"], crop_info["crop_right"])
        front_frame_bgr, rear_frame_bgr = split_front_rear_frame(cropped_frame_bgr)
        for view, frame_bgr in [("front", front_frame_bgr), ("rear", rear_frame_bgr)]:
            state = states[view]
            current_gray = content_update.prepare_gray_frame(
                frame_bgr,
                resize_width=CONTENT_UPDATE_FRAME_THINNING_RESIZE_WIDTH,
                gaussian_blur_kernel=CONTENT_UPDATE_FRAME_THINNING_GAUSSIAN_BLUR_KERNEL,
            )
            previous_gray = state["previous_gray"]
            if previous_gray is not None:
                tile_stats = content_update.frame_tile_abs_diff_stats(
                    current_gray,
                    previous_gray,
                    diff_threshold=diff_threshold,
                    grid_rows=CONTENT_UPDATE_TILE_GRID_ROWS,
                    grid_cols=CONTENT_UPDATE_TILE_GRID_COLS,
                    top_fraction=CONTENT_UPDATE_TILE_TOP_FRACTION,
                    active_threshold_ratio=CONTENT_UPDATE_TILE_ACTIVE_THRESHOLD_RATIO,
                    min_active_tiles=CONTENT_UPDATE_TILE_MIN_ACTIVE_TILES,
                    min_active_ratio=CONTENT_UPDATE_TILE_MIN_ACTIVE_RATIO,
                )
                state["diff_by_ordinal"][ordinal] = float(tile_stats["score"])
            state["previous_gray"] = current_gray

    def find_peak_ordinals(diff_by_ordinal: np.ndarray) -> tuple[list[int], float]:
        finite_scores = diff_by_ordinal[np.isfinite(diff_by_ordinal)]
        if finite_scores.size == 0:
            return [], np.nan
        percentile_threshold = float(np.percentile(finite_scores, peak_score_percentile))
        score_threshold = max(float(diff_threshold), percentile_threshold)
        peaks: list[int] = []
        for ordinal in range(1, input_frame_pairs):
            score = float(diff_by_ordinal[ordinal])
            if not np.isfinite(score) or score < score_threshold:
                continue
            previous_score = float(diff_by_ordinal[ordinal - 1]) if ordinal - 1 >= 1 else -np.inf
            next_score = float(diff_by_ordinal[ordinal + 1]) if ordinal + 1 < input_frame_pairs else -np.inf
            if score >= previous_score and score >= next_score and (score > previous_score or score > next_score):
                peaks.append(int(ordinal))
        return peaks, score_threshold

    def local_average_peak_intervals_frames(peak_ordinals: list[int]) -> np.ndarray:
        ordinals = np.asarray(sorted(set(int(ordinal) for ordinal in peak_ordinals)), dtype=float)
        if ordinals.size < 2:
            return np.zeros((0,), dtype=float)
        window_updates = min(base_interval_window_updates, int(ordinals.size))
        if window_updates < 2:
            return np.zeros((0,), dtype=float)
        intervals = (ordinals[window_updates - 1:] - ordinals[:-(window_updates - 1)]) / float(window_updates - 1)
        return intervals[np.isfinite(intervals) & (intervals > 0.0)]

    for state in states.values():
        peak_ordinals, score_threshold = find_peak_ordinals(state["diff_by_ordinal"])
        state["peak_ordinals"] = peak_ordinals
        state["peak_score_threshold"] = float(score_threshold) if np.isfinite(score_threshold) else np.nan

    for state in states.values():
        base_interval_candidates_frames = local_average_peak_intervals_frames(state["peak_ordinals"])
        base_interval_frames = np.nan
        if base_interval_candidates_frames.size:
            base_interval_frames = float(np.percentile(base_interval_candidates_frames, base_interval_percentile))
        grid_interval_frames = int(round(base_interval_frames)) if np.isfinite(base_interval_frames) and base_interval_frames > 0 else fallback_interval_frames
        grid_interval_frames = max(1, int(grid_interval_frames))
        grid_interval_sec = float(grid_interval_frames / fallback_fps) if fallback_fps > 0 else np.nan
        base_interval_sec = float(base_interval_frames / fallback_fps) if fallback_fps > 0 and np.isfinite(base_interval_frames) else np.nan
        grid_ordinals = list(range(0, input_frame_pairs, grid_interval_frames))
        if CONTENT_UPDATE_GRID_INCLUDE_LAST_FRAME and input_frame_pairs > 0 and (input_frame_pairs - 1) not in grid_ordinals:
            grid_ordinals.append(input_frame_pairs - 1)
        state["grid_ordinals"] = sorted(set(int(ordinal) for ordinal in grid_ordinals if 0 <= int(ordinal) < input_frame_pairs))
        state["grid_base_interval_frames"] = float(base_interval_frames) if np.isfinite(base_interval_frames) else np.nan
        state["grid_base_interval_sec"] = float(base_interval_sec) if np.isfinite(base_interval_sec) else np.nan
        state["grid_interval_frames"] = int(grid_interval_frames)
        state["grid_interval_sec"] = float(grid_interval_sec) if np.isfinite(grid_interval_sec) else np.nan

    def choose_representative_ordinals(
        grid_ordinals: list[int],
        peak_ordinals: list[int],
        diff_by_ordinal: np.ndarray,
        grid_interval_frames: int,
    ) -> list[int]:
        peaks = np.asarray(sorted(set(int(ordinal) for ordinal in peak_ordinals)), dtype=int)
        search_radius = max(1, int(round(grid_interval_frames / 2)))
        representatives: list[int] = []
        used: set[int] = set()
        for grid_ordinal in grid_ordinals:
            chosen: int | None = None
            if peaks.size:
                distances = np.abs(peaks - int(grid_ordinal))
                peak_candidates = [
                    (int(distances[position]), int(peaks[position]))
                    for position in range(int(peaks.size))
                    if int(distances[position]) <= search_radius and int(peaks[position]) not in used
                ]
                if peak_candidates:
                    chosen = min(peak_candidates)[1]
            if chosen is None:
                window_start = max(0, int(grid_ordinal) - search_radius)
                window_end = min(input_frame_pairs - 1, int(grid_ordinal) + search_radius)
                diff_candidates = []
                for candidate in range(window_start, window_end + 1):
                    score = float(diff_by_ordinal[candidate])
                    if np.isfinite(score) and candidate not in used:
                        diff_candidates.append((-score, abs(candidate - int(grid_ordinal)), candidate))
                if diff_candidates:
                    chosen = min(diff_candidates)[2]
            if chosen is None:
                chosen = int(grid_ordinal)
            if chosen in used:
                for offset in range(1, search_radius + 1):
                    for candidate in (int(grid_ordinal) - offset, int(grid_ordinal) + offset):
                        if 0 <= candidate < input_frame_pairs and candidate not in used:
                            chosen = candidate
                            break
                    if chosen not in used:
                        break
            if 0 <= chosen < input_frame_pairs and chosen not in used:
                representatives.append(chosen)
                used.add(chosen)
        return sorted(representatives)

    def compute_max_removed_run(kept_ordinals: set[int]) -> int:
        max_run = 0
        run = 0
        for ordinal in range(input_frame_pairs):
            if ordinal in kept_ordinals:
                max_run = max(max_run, run)
                run = 0
            else:
                run += 1
        return int(max(max_run, run))

    for state in states.values():
        final_ordinals = choose_representative_ordinals(
            state["grid_ordinals"],
            state["peak_ordinals"],
            state["diff_by_ordinal"],
            int(state["grid_interval_frames"]),
        )
        final_ordinal_set = set(final_ordinals)
        max_interval_frames = max(
            (int(next_ordinal) - int(previous_ordinal) for previous_ordinal, next_ordinal in zip(final_ordinals, final_ordinals[1:])),
            default=0,
        )
        state["final_ordinals"] = final_ordinals
        state["kept"] = [int(source_frame_indices[ordinal]) for ordinal in final_ordinals]
        state["duplicate_count"] = int(max(0, input_frame_pairs - len(final_ordinals)))
        state["max_duplicate_run"] = compute_max_removed_run(final_ordinal_set)
        state["max_time_since_update"] = float(max_interval_frames / fallback_fps) if fallback_fps > 0 else 0.0

    kept_by_view = {view: list(state["kept"]) for view, state in states.items()}

    info = base_info()
    for view, state in states.items():
        finite_diff_scores = state["diff_by_ordinal"][np.isfinite(state["diff_by_ordinal"])]
        info.update(content_update.summarize_content_update_view(
            f"{view}_content_update",
            enabled=enabled,
            input_frame_pairs=input_frame_pairs,
            kept_indices=list(state["kept"]),
            duplicate_count=int(state["duplicate_count"]),
            diff_scores=finite_diff_scores.tolist(),
            max_duplicate_run=int(state["max_duplicate_run"]),
            max_time_since_content_update=float(state["max_time_since_update"]),
            **summary_kwargs,
        ))
        info[f"{view}_content_update_peak_frame_pairs"] = int(len(state["peak_ordinals"]))
        info[f"{view}_content_update_grid_frame_pairs"] = int(len(state["grid_ordinals"]))
        info[f"{view}_content_update_peak_score_threshold"] = float(state["peak_score_threshold"])
        info[f"{view}_content_update_grid_base_interval_frames"] = float(state["grid_base_interval_frames"])
        info[f"{view}_content_update_grid_base_interval_sec"] = float(state["grid_base_interval_sec"])
        info[f"{view}_content_update_grid_interval_frames"] = int(state["grid_interval_frames"])
        info[f"{view}_content_update_grid_interval_sec"] = float(state["grid_interval_sec"])
    return kept_by_view, info


# 関数メモ: load_outdoor_video_names
# - outdoor_list.txt を読み、屋外動画として扱う元ファイル名の集合を作ります。
# - ファイル配置だけでは indoor/outdoor が分からない場合の環境ラベル付けに使います。

def load_outdoor_video_names(outdoor_list_path: Path = OUTDOOR_LIST_PATH) -> set[str]:
    if not outdoor_list_path.exists():
        return set()
    return {line.strip() for line in outdoor_list_path.read_text(encoding="utf-8").splitlines() if line.strip()}


# 関数メモ: load_frame_cut_config
# - frame_cut.txt から、動画ごとの切り出し開始・終了指定と任意の重複判定閾値を読みます。
# - 空行と `#` で始まる行はコメント行として読み飛ばします。
# - `all` 行があれば全動画共通のカット数として扱い、なければ `all 0 0` として扱います。
# - 4列目 duplicate_diff_threshold は個別動画行だけで有効です。省略時は DUPLICATE_DIFF_THRESHOLD を使います。
# - 手動で指定した trim 範囲を自動前処理に反映するための設定読み込みです。

def load_frame_cut_config(frame_cut_path: Path = FRAME_CUT_PATH) -> dict[str, dict[str, Any]]:
    if not frame_cut_path.exists():
        return {"all": {"front": 0, "rear": 0}}

    frame_cut_config: dict[str, dict[str, Any]] = {"all": {"front": 0, "rear": 0}}
    for line_number, raw_line in enumerate(frame_cut_path.read_text(encoding="utf-8").splitlines(), start=1):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split()
        if len(parts) not in {3, 4}:
            raise ValueError(f"Invalid frame_cut.txt line {line_number}: {raw_line!r}")

        video_name, front_cut_text, rear_cut_text = parts[:3]
        video_name = "all" if video_name.lower() == "all" else video_name
        try:
            front_cut_frames = int(front_cut_text)
            rear_cut_frames = int(rear_cut_text)
        except ValueError as error:
            raise ValueError(f"Invalid frame_cut.txt frame count at line {line_number}: {raw_line!r}") from error

        duplicate_diff_threshold = None
        if len(parts) == 4:
            if video_name == "all":
                raise ValueError(f"frame_cut.txt duplicate_diff_threshold cannot be set on all line {line_number}: {raw_line!r}")
            try:
                duplicate_diff_threshold = float(parts[3])
            except ValueError as error:
                raise ValueError(f"Invalid frame_cut.txt duplicate_diff_threshold at line {line_number}: {raw_line!r}") from error
            if duplicate_diff_threshold < 0:
                raise ValueError(f"frame_cut.txt duplicate_diff_threshold must be non-negative at line {line_number}: {raw_line!r}")

        if front_cut_frames < 0 or rear_cut_frames < 0:
            raise ValueError(f"frame_cut.txt frame counts must be non-negative at line {line_number}: {raw_line!r}")

        entry: dict[str, Any] = {
            "front": front_cut_frames,
            "rear": rear_cut_frames,
        }
        if duplicate_diff_threshold is not None:
            entry["duplicate_diff_threshold"] = float(duplicate_diff_threshold)
        frame_cut_config[video_name] = entry
    return frame_cut_config


# 関数メモ: resolve_frame_cut_for_video
# - `all` の共通カット数と個別動画のカット数を加算し、その動画で使う実効 frame_cut を返します。
# - 個別動画行の4列目があれば、その動画だけ内容更新フレーム間引き閾値を上書きします。
# - ここで合算した値を manifest 記録や既存成果物のスキップ判定にも使います。

def resolve_frame_cut_for_video(
    video_name: str,
    frame_cut_config: dict[str, dict[str, Any]] | None,
) -> dict[str, int | float | bool]:
    config = frame_cut_config or {}
    all_cut = config.get("all", {})
    individual_cut = config.get(video_name, {})

    all_front_frames = int(all_cut.get("front", 0))
    all_rear_frames = int(all_cut.get("rear", 0))
    individual_front_frames = int(individual_cut.get("front", 0))
    individual_rear_frames = int(individual_cut.get("rear", 0))
    individual_duplicate_diff_threshold = individual_cut.get("duplicate_diff_threshold")
    duplicate_diff_threshold = (
        float(individual_duplicate_diff_threshold)
        if individual_duplicate_diff_threshold is not None
        else float(DUPLICATE_DIFF_THRESHOLD)
    )

    return {
        "front": all_front_frames + individual_front_frames,
        "rear": all_rear_frames + individual_rear_frames,
        "all_front": all_front_frames,
        "all_rear": all_rear_frames,
        "individual_front": individual_front_frames,
        "individual_rear": individual_rear_frames,
        "has_global_config": bool(all_front_frames or all_rear_frames),
        "duplicate_diff_threshold": duplicate_diff_threshold,
        "has_individual_duplicate_diff_threshold": individual_duplicate_diff_threshold is not None,
        "has_individual_config": video_name in config,
    }


# 関数メモ: find_movie_preprocess_targets
# - 元動画ディレクトリを走査し、前処理対象の動画と category/environment を一覧化します。
# - バッチ処理本体がこのリストだけを順に処理すればよい形にします。

def find_movie_preprocess_targets() -> list[dict[str, Any]]:
    targets: list[dict[str, Any]] = []
    outdoor_video_names = load_outdoor_video_names()
    for category, video_dir in INPUT_VIDEO_DIRS.items():
        if not video_dir.exists():
            continue
        for video_path in sorted(video_dir.glob("*.mp4")):
            if video_path.is_file():
                targets.append({
                    "category": category,
                    "environment": "outdoor" if video_path.stem in outdoor_video_names else "indoor",
                    "video_path": video_path,
                })
    return targets


# 関数メモ: normalize_movie_preprocess_target_video_names
# - 設定セルの動画名指定を、stem / ファイル名どちらでも照合しやすい集合に正規化します。
# - None / 空文字 / 空リストの場合は全動画処理を表す空集合を返します。

def normalize_movie_preprocess_target_video_names(value: Any) -> set[str]:
    if value is None:
        return set()
    values = [value] if isinstance(value, (str, Path)) else list(value)
    names: set[str] = set()
    for raw_name in values:
        if raw_name is None:
            continue
        text = str(raw_name).strip()
        if not text:
            continue
        path = Path(text)
        names.add(path.name)
        names.add(path.stem)
    return names


# 関数メモ: filter_movie_preprocess_targets_by_video_name
# - 動画名指定がある場合だけ前処理対象を絞ります。未指定なら元の全件リストを返します。
# - 指定名は `foo` と `foo.mp4` のどちらでも照合できます。

def filter_movie_preprocess_targets_by_video_name(
    targets: list[dict[str, Any]],
    target_video_names: Any = None,
) -> tuple[list[dict[str, Any]], set[str]]:
    if target_video_names is None:
        target_video_names = MOVIE_PREPROCESS_TARGET_VIDEO_NAMES
    requested_names = normalize_movie_preprocess_target_video_names(target_video_names)
    if not requested_names:
        return list(targets), set()
    selected_targets = [
        target for target in targets
        if Path(target["video_path"]).name in requested_names
        or Path(target["video_path"]).stem in requested_names
    ]
    if not selected_targets:
        available = ", ".join(sorted(Path(target["video_path"]).name for target in targets)[:20])
        raise ValueError(f"MOVIE_PREPROCESS_TARGET_VIDEO_NAMES did not match any input videos: {sorted(requested_names)}. available examples: {available}")
    return selected_targets, requested_names


def movie_preprocess_manifest_row_key(row: pd.Series | dict[str, Any]) -> tuple[str, str]:
    return (str(row.get("category", "")), Path(str(row.get("input_video_path", ""))).stem)


# 関数メモ: build_movie_preprocess_output_paths
# - 入力動画から front/rear/audio/manifest 用の出力パスを組み立てます。
# - category/environment/sample_id の命名を1か所に集約し、保存先の揺れを防ぎます。

def build_movie_preprocess_output_paths(
    video_path: Path,
    category: str,
    environment: str,
    output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR,
) -> tuple[Path, Path, Path]:
    output_dir = output_base_dir / category / environment
    front_output_path = output_dir / f"{video_path.stem}_front{video_path.suffix}"
    rear_output_path = output_dir / f"{video_path.stem}_rear{video_path.suffix}"
    audio_output_path = output_dir / f"{video_path.stem}_audio.wav"
    return front_output_path, rear_output_path, audio_output_path


# 関数メモ: find_existing_movie_preprocess_files
# - 同じ sample_id に対する既存の前処理成果物を探します。
# - 再処理や削除が必要かを判断するため、現在の出力状態を確認します。

def find_existing_movie_preprocess_files(
    video_name: str,
    output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR,
) -> list[Path]:
    if not output_base_dir.exists():
        return []

    existing_paths = [
        path
        for path in output_base_dir.rglob("*")
        if path.is_file()
        and path.name != MOVIE_PREPROCESS_MANIFEST_PATH.name
        and (path.stem == video_name or path.stem.startswith((f"{video_name}_", f"{video_name}-")))
    ]
    return sorted(existing_paths)


# 関数メモ: load_movie_preprocess_manifest
# - 過去の前処理 manifest を読み込みます。
# - 存在しない場合は空の表を返し、初回実行と再実行を同じ流れで扱います。

def load_movie_preprocess_manifest(
    manifest_path: Path = MOVIE_PREPROCESS_MANIFEST_PATH,
) -> pd.DataFrame:
    if not manifest_path.exists():
        return pd.DataFrame()
    return pd.read_csv(manifest_path)


# 関数メモ: build_previous_frame_cut_records
# - 過去 manifest から、動画ごとの frame cut 設定や出力状態を引ける辞書を作ります。
# - 設定が変わった動画だけ再処理するための比較材料です。

def build_previous_frame_cut_records(manifest_df: pd.DataFrame) -> dict[str, dict[str, Any]]:
    required_columns = {"input_video_path", "frame_cut_front_frames", "frame_cut_rear_frames"}
    if manifest_df.empty or not required_columns.issubset(manifest_df.columns):
        return {}

    records: dict[str, dict[str, Any]] = {}
    for row in manifest_df.to_dict("records"):
        if pd.isna(row.get("frame_cut_front_frames")) or pd.isna(row.get("frame_cut_rear_frames")):
            continue
        video_name = Path(str(row["input_video_path"])).stem
        record: dict[str, Any] = {
            "front": int(row["frame_cut_front_frames"]),
            "rear": int(row["frame_cut_rear_frames"]),
        }
        for column in [
            "content_update_frame_thinning_enabled",
            "content_update_duplicate_diff_threshold",
            "content_update_resize_width",
            "content_update_static_scene_min_sec",
            "content_update_flow_target_dt",
            "content_update_grid_peak_score_percentile",
            "content_update_grid_base_interval_window_updates",
            "content_update_grid_base_interval_percentile",
            "content_update_grid_fallback_interval_sec",
            "content_update_grid_include_last_frame",
            "content_update_flow_normalize_by_dt",
        ]:
            value = row.get(column)
            if value is not None and not pd.isna(value):
                record[column] = value
        records[video_name] = record
    return records


# 関数メモ: frame_cut_record_matches
# - 今回の frame cut 設定が、過去 manifest に記録された設定と一致するかを判定します。
# - trim 指定が変わったときに古い成果物を使い回さないためのチェックです。

def frame_cut_record_matches(
    previous_record: dict[str, Any] | None,
    front_cut_frames: int,
    rear_cut_frames: int,
) -> bool:
    if previous_record is None:
        return False
    return previous_record.get("front") == front_cut_frames and previous_record.get("rear") == rear_cut_frames


# 関数メモ: manifest_bool
# - manifest 上の文字列・数値・真偽値を bool として安定的に解釈します。
# - CSV 由来の `True` / `False` / `1` / `0` の揺れを吸収します。

def manifest_bool(value: Any) -> bool:
    if isinstance(value, str):
        return value.strip().lower() in {"1", "true", "yes"}
    return bool(value)


# 関数メモ: content_update_frame_thinning_record_matches
# - 内容更新フレーム間引きに関する今回設定と過去設定が一致するかを判定します。
# - 既存のmovie_preprocess出力を、間引き設定変更後に誤って再利用しないための比較です。

def content_update_frame_thinning_record_matches(
    previous_record: dict[str, Any] | None,
    enabled: bool = CONTENT_UPDATE_FRAME_THINNING_ENABLED,
    diff_threshold: float = DUPLICATE_DIFF_THRESHOLD,
    resize_width: int | None = CONTENT_UPDATE_FRAME_THINNING_RESIZE_WIDTH,
    static_scene_min_sec: float = STATIC_SCENE_MIN_SEC,
    flow_target_dt: float = FLOW_TARGET_DT,
    tile_grid_rows: int = CONTENT_UPDATE_TILE_GRID_ROWS,
    tile_grid_cols: int = CONTENT_UPDATE_TILE_GRID_COLS,
    tile_top_fraction: float = CONTENT_UPDATE_TILE_TOP_FRACTION,
    tile_active_threshold_ratio: float = CONTENT_UPDATE_TILE_ACTIVE_THRESHOLD_RATIO,
    tile_min_active_tiles: int = CONTENT_UPDATE_TILE_MIN_ACTIVE_TILES,
    tile_min_active_ratio: float = CONTENT_UPDATE_TILE_MIN_ACTIVE_RATIO,
    grid_peak_score_percentile: float = CONTENT_UPDATE_GRID_PEAK_SCORE_PERCENTILE,
    grid_base_interval_window_updates: int = CONTENT_UPDATE_GRID_BASE_INTERVAL_WINDOW_UPDATES,
    grid_base_interval_percentile: float = CONTENT_UPDATE_GRID_BASE_INTERVAL_PERCENTILE,
    grid_fallback_interval_sec: float = CONTENT_UPDATE_GRID_FALLBACK_INTERVAL_SEC,
    grid_include_last_frame: bool = CONTENT_UPDATE_GRID_INCLUDE_LAST_FRAME,
    flow_normalize_by_dt: bool = FLOW_NORMALIZE_BY_DT,
) -> bool:
    if not enabled:
        return True
    if previous_record is None or not manifest_bool(previous_record.get("content_update_frame_thinning_enabled", False)):
        return False

    expected_resize_width = -1 if resize_width is None else int(resize_width)
    try:
        return (
            np.isclose(float(previous_record.get("content_update_duplicate_diff_threshold")), float(diff_threshold))
            and int(float(previous_record.get("content_update_resize_width"))) == expected_resize_width
            and np.isclose(float(previous_record.get("content_update_static_scene_min_sec")), float(static_scene_min_sec))
            and np.isclose(float(previous_record.get("content_update_flow_target_dt")), float(flow_target_dt))
            and int(float(previous_record.get("content_update_tile_grid_rows"))) == int(tile_grid_rows)
            and int(float(previous_record.get("content_update_tile_grid_cols"))) == int(tile_grid_cols)
            and np.isclose(float(previous_record.get("content_update_tile_top_fraction")), float(tile_top_fraction))
            and np.isclose(float(previous_record.get("content_update_tile_active_threshold_ratio")), float(tile_active_threshold_ratio))
            and int(float(previous_record.get("content_update_tile_min_active_tiles"))) == int(tile_min_active_tiles)
            and np.isclose(float(previous_record.get("content_update_tile_min_active_ratio")), float(tile_min_active_ratio))
            and np.isclose(float(previous_record.get("content_update_grid_peak_score_percentile")), float(grid_peak_score_percentile))
            and int(float(previous_record.get("content_update_grid_base_interval_window_updates"))) == int(grid_base_interval_window_updates)
            and np.isclose(float(previous_record.get("content_update_grid_base_interval_percentile")), float(grid_base_interval_percentile))
            and np.isclose(float(previous_record.get("content_update_grid_fallback_interval_sec")), float(grid_fallback_interval_sec))
            and manifest_bool(previous_record.get("content_update_grid_include_last_frame")) == bool(grid_include_last_frame)
            and manifest_bool(previous_record.get("content_update_flow_normalize_by_dt")) == bool(flow_normalize_by_dt)
        )
    except (TypeError, ValueError):
        return False


# 関数メモ: delete_movie_preprocess_files
# - 指定された前処理成果物を削除します。
# -  stale な出力を消してから再処理することで、古いファイルの混入を防ぎます。

def delete_movie_preprocess_files(paths: list[Path]) -> None:
    for path in paths:
        if path.exists():
            path.unlink()


# 関数メモ: expected_movie_preprocess_output_paths
# - manifest の1行から、本来存在すべき前処理成果物パスを復元します。
# - 現在のディレクトリ内にある余計なファイルを判定する基準にします。

def expected_movie_preprocess_output_paths(
    targets: list[dict[str, Any]],
    output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR,
) -> set[Path]:
    expected_paths: set[Path] = set()
    for target in targets:
        expected_paths.update(
            path.resolve()
            for path in build_movie_preprocess_output_paths(
                video_path=target["video_path"],
                category=target["category"],
                environment=target["environment"],
                output_base_dir=output_base_dir,
            )
        )
    return expected_paths


# 関数メモ: iter_movie_preprocess_artifacts
# - movie_preprocess 配下の mp4 / wav / csv など成果物ファイルを列挙します。
# - 不要ファイル掃除や状態確認を一括で行うための走査関数です。

def iter_movie_preprocess_artifacts(output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR) -> list[Path]:
    artifacts: list[Path] = []
    for category in INPUT_VIDEO_DIRS:
        category_dir = output_base_dir / category
        if not category_dir.exists():
            continue
        for path in category_dir.glob("*/*"):
            if not path.is_file():
                continue
            if path.name.endswith(("_front.mp4", "_rear.mp4", "_audio.wav")):
                artifacts.append(path)
    return sorted(artifacts)


# 関数メモ: remove_empty_movie_preprocess_dirs
# - 成果物削除後に空になったカテゴリ・環境ディレクトリを削除します。
# - 再実行を繰り返しても出力ツリーに空ディレクトリが残りにくくします。

def remove_empty_movie_preprocess_dirs(output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR) -> None:
    for directory in sorted(
        [p for p in output_base_dir.glob("*/*") if p.is_dir()],
        key=lambda p: len(p.parts),
        reverse=True,
    ):
        try:
            directory.rmdir()
        except OSError:
            pass


# 関数メモ: cleanup_stale_movie_preprocess_outputs
# - 今回の manifest から期待されない古い成果物を削除します。
# - 入力動画や設定が変わったあと、古い前処理結果を誤って使わないための掃除です。

def cleanup_stale_movie_preprocess_outputs(
    targets: list[dict[str, Any]],
    output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR,
    delete_stale: bool = MOVIE_PREPROCESS_DELETE_STALE,
) -> list[Path]:
    expected_paths = expected_movie_preprocess_output_paths(targets, output_base_dir=output_base_dir)
    stale_paths = [
        path for path in iter_movie_preprocess_artifacts(output_base_dir=output_base_dir)
        if path.resolve() not in expected_paths
    ]
    if not stale_paths:
        print("stale movie_preprocess artifacts: 0")
        return []

    action = "delete" if delete_stale else "keep"
    print(f"stale movie_preprocess artifacts: {len(stale_paths)} ({action})")
    for path in stale_paths:
        print("  stale:", path)
        if delete_stale:
            path.unlink()

    if delete_stale:
        remove_empty_movie_preprocess_dirs(output_base_dir=output_base_dir)
    return stale_paths


# 関数メモ: has_audio_stream
# - ffprobe で動画に音声ストリームが含まれるかを確認します。
# - 音声がない動画に対して ffmpeg 抽出を実行して失敗扱いにするのを避けます。

def has_audio_stream(video_path: Path) -> bool:
    command = [
        FFPROBE_PATH,
        "-v",
        "error",
        "-select_streams",
        "a",
        "-show_entries",
        "stream=index",
        "-of",
        "csv=p=0",
        str(video_path),
    ]
    completed = subprocess.run(command, check=False, capture_output=True, text=True)
    return bool(completed.stdout.strip())


# 関数メモ: extract_trimmed_audio_to_wav
# - ffmpeg を使い、動画の指定区間からモノラル wav 音声を抽出します。
# - front/rear 動画と同じ trim 範囲に合わせ、音声特徴の時刻を動画とそろえます。

def extract_trimmed_audio_to_wav(
    video_path: Path,
    output_wav_path: Path,
    start_sec: float,
    duration_sec: float,
    overwrite: bool = MOVIE_PREPROCESS_OVERWRITE,
    sample_rate: int = AUDIO_SAMPLE_RATE,
    channels: int = AUDIO_MONO_CHANNELS,
) -> str:
    if duration_sec <= 0:
        return "skipped_zero_duration"
    if not overwrite and output_wav_path.exists():
        return "skipped_existing_audio"
    if not has_audio_stream(video_path):
        return "skipped_missing_audio_stream"

    ensure_parent_directory(output_wav_path)
    command = [
        FFMPEG_PATH,
        "-hide_banner",
        "-loglevel",
        "error",
        "-y",
        "-ss",
        f"{max(start_sec, 0.0):.6f}",
        "-i",
        str(video_path),
        "-t",
        f"{max(duration_sec, 0.0):.6f}",
        "-vn",
        "-ac",
        str(channels),
        "-ar",
        str(sample_rate),
        "-c:a",
        "pcm_s16le",
        str(output_wav_path),
    ]
    subprocess.run(command, check=True)
    return "written"


## 2. 前処理実行


In [180]:
# ------------------------------------------------------------
# セル概要: 前処理本体とバッチ実行
# ------------------------------------------------------------
# - 1本の元動画を前後カメラ動画と音声wavへ変換し、その処理をデータセット全体へ適用します。
# - 前回 manifest と設定を比較し、再処理が必要なものだけを処理できるようにしています。
# ------------------------------------------------------------

# 関数メモ: preprocess_movie_to_front_rear_audio
# - 1本の元動画を読み、trim・黒帯除去・front/rear分割・音声抽出・manifest行作成まで行います。
# - この関数が動画前処理の中心で、成功・skip・error の状態もここで決まります。

def preprocess_movie_to_front_rear_audio(
    video_path: Path,
    category: str,
    environment: str,
    output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR,
    overwrite: bool = MOVIE_PREPROCESS_OVERWRITE,
    frame_cut_config: dict[str, dict[str, Any]] | None = None,
    previous_frame_cut_records: dict[str, dict[str, Any]] | None = None,
) -> dict[str, Any]:
    metadata = read_video_metadata(video_path)
    front_output_path, rear_output_path, audio_output_path = build_movie_preprocess_output_paths(
        video_path=video_path,
        category=category,
        environment=environment,
        output_base_dir=output_base_dir,
    )

    result: dict[str, Any] = {
        "category": category,
        "environment": environment,
        "input_video_path": str(video_path),
        "front_output_path": str(front_output_path),
        "rear_output_path": str(rear_output_path),
        "audio_output_path": str(audio_output_path),
        "input_frame_count": int(metadata["frame_count"]),
        "input_fps": float(metadata["fps"]),
        "input_width": int(metadata["width"]),
        "input_height": int(metadata["height"]),
        "input_duration_sec": float(metadata["duration_sec"]),
        "content_update_frame_thinning_enabled": bool(CONTENT_UPDATE_FRAME_THINNING_ENABLED),
        "content_update_duplicate_diff_threshold": float(DUPLICATE_DIFF_THRESHOLD),
        "content_update_resize_width": -1 if CONTENT_UPDATE_FRAME_THINNING_RESIZE_WIDTH is None else int(CONTENT_UPDATE_FRAME_THINNING_RESIZE_WIDTH),
        "content_update_static_scene_min_sec": float(STATIC_SCENE_MIN_SEC),
        "content_update_tile_grid_rows": int(CONTENT_UPDATE_TILE_GRID_ROWS),
        "content_update_tile_grid_cols": int(CONTENT_UPDATE_TILE_GRID_COLS),
        "content_update_tile_top_fraction": float(CONTENT_UPDATE_TILE_TOP_FRACTION),
        "content_update_tile_active_threshold_ratio": float(CONTENT_UPDATE_TILE_ACTIVE_THRESHOLD_RATIO),
        "content_update_tile_min_active_tiles": int(CONTENT_UPDATE_TILE_MIN_ACTIVE_TILES),
        "content_update_tile_min_active_ratio": float(CONTENT_UPDATE_TILE_MIN_ACTIVE_RATIO),
        "content_update_flow_target_dt": float(FLOW_TARGET_DT),
        "content_update_grid_peak_score_percentile": float(CONTENT_UPDATE_GRID_PEAK_SCORE_PERCENTILE),
        "content_update_grid_base_interval_window_updates": int(CONTENT_UPDATE_GRID_BASE_INTERVAL_WINDOW_UPDATES),
        "content_update_grid_base_interval_percentile": float(CONTENT_UPDATE_GRID_BASE_INTERVAL_PERCENTILE),
        "content_update_grid_fallback_interval_sec": float(CONTENT_UPDATE_GRID_FALLBACK_INTERVAL_SEC),
        "content_update_grid_include_last_frame": bool(CONTENT_UPDATE_GRID_INCLUDE_LAST_FRAME),
        "content_update_flow_normalize_by_dt": bool(FLOW_NORMALIZE_BY_DT),
    }

    processing_fps = float(metadata["fps"]) if float(metadata["fps"]) > 0 else 30.0
    processing_frame_count = int(metadata["frame_count"])
    source_frame_indices = list(range(processing_frame_count))
    result.update({
        "frame_cut_input_frame_count": int(processing_frame_count),
        "frame_cut_input_fps": float(processing_fps),
    })

    frame_cut = resolve_frame_cut_for_video(video_path.stem, frame_cut_config)
    frame_cut_has_explicit_config = bool(
        frame_cut["has_global_config"] or frame_cut["has_individual_config"]
    )
    front_cut_frames = int(frame_cut["front"])
    rear_cut_frames = int(frame_cut["rear"])
    content_update_duplicate_diff_threshold = float(frame_cut["duplicate_diff_threshold"])
    frame_cut_start_index = front_cut_frames
    frame_cut_end_index = processing_frame_count - rear_cut_frames
    remaining_after_initial_frame_cut = max(frame_cut_end_index - frame_cut_start_index, 0)
    frame_cut_source_frame_indices = source_frame_indices[
        frame_cut_start_index:max(frame_cut_start_index, frame_cut_end_index)
    ]
    result.update({
        "frame_cut_front_frames": front_cut_frames,
        "frame_cut_rear_frames": rear_cut_frames,
        "frame_cut_all_front_frames": int(frame_cut["all_front"]),
        "frame_cut_all_rear_frames": int(frame_cut["all_rear"]),
        "frame_cut_individual_front_frames": int(frame_cut["individual_front"]),
        "frame_cut_individual_rear_frames": int(frame_cut["individual_rear"]),
        "frame_cut_has_global_config": bool(frame_cut["has_global_config"]),
        "frame_cut_has_individual_config": bool(frame_cut["has_individual_config"]),
        "frame_cut_duplicate_diff_threshold": float(content_update_duplicate_diff_threshold),
        "frame_cut_has_individual_duplicate_diff_threshold": bool(frame_cut["has_individual_duplicate_diff_threshold"]),
        "content_update_duplicate_diff_threshold": float(content_update_duplicate_diff_threshold),
        "frame_cut_start_frame_index": int(frame_cut_start_index),
        "frame_cut_end_frame_index": int(max(frame_cut_start_index, frame_cut_end_index)),
        "remaining_frame_pairs_after_frame_cut": int(remaining_after_initial_frame_cut),
    })

    existing_preprocess_files = find_existing_movie_preprocess_files(
        video_path.stem,
        output_base_dir=output_base_dir,
    )
    previous_frame_cut_record = (previous_frame_cut_records or {}).get(video_path.stem)
    if previous_frame_cut_record is not None:
        result.update({
            "previous_frame_cut_front_frames": int(previous_frame_cut_record["front"]),
            "previous_frame_cut_rear_frames": int(previous_frame_cut_record["rear"]),
        })
        if "content_update_frame_thinning_enabled" in previous_frame_cut_record:
            result.update({
                "previous_content_update_frame_thinning_enabled": manifest_bool(
                    previous_frame_cut_record.get("content_update_frame_thinning_enabled")
                ),
                "previous_content_update_duplicate_diff_threshold": previous_frame_cut_record.get(
                    "content_update_duplicate_diff_threshold"
                ),
                "previous_content_update_resize_width": previous_frame_cut_record.get("content_update_resize_width"),
                "previous_content_update_tile_grid_rows": previous_frame_cut_record.get("content_update_tile_grid_rows"),
                "previous_content_update_tile_grid_cols": previous_frame_cut_record.get("content_update_tile_grid_cols"),
                "previous_content_update_tile_top_fraction": previous_frame_cut_record.get(
                    "content_update_tile_top_fraction"
                ),
                "previous_content_update_tile_active_threshold_ratio": previous_frame_cut_record.get(
                    "content_update_tile_active_threshold_ratio"
                ),
                "previous_content_update_tile_min_active_tiles": previous_frame_cut_record.get(
                    "content_update_tile_min_active_tiles"
                ),
                "previous_content_update_tile_min_active_ratio": previous_frame_cut_record.get(
                    "content_update_tile_min_active_ratio"
                ),
                "previous_content_update_grid_peak_score_percentile": previous_frame_cut_record.get(
                    "content_update_grid_peak_score_percentile"
                ),
                "previous_content_update_grid_base_interval_window_updates": previous_frame_cut_record.get(
                    "content_update_grid_base_interval_window_updates"
                ),
                "previous_content_update_grid_base_interval_percentile": previous_frame_cut_record.get(
                    "content_update_grid_base_interval_percentile"
                ),
                "previous_content_update_grid_fallback_interval_sec": previous_frame_cut_record.get(
                    "content_update_grid_fallback_interval_sec"
                ),
                "previous_content_update_grid_include_last_frame": previous_frame_cut_record.get(
                    "content_update_grid_include_last_frame"
                ),
            })

    if not overwrite and existing_preprocess_files:
        result["existing_preprocess_files"] = ";".join(str(path) for path in existing_preprocess_files)
        frame_cut_matches = frame_cut_record_matches(previous_frame_cut_record, front_cut_frames, rear_cut_frames)
        content_update_matches = content_update_frame_thinning_record_matches(
            previous_frame_cut_record,
            diff_threshold=content_update_duplicate_diff_threshold,
        )
        if frame_cut_matches and content_update_matches:
            result.update({
                "status": "skipped_existing_preprocess_file",
                "audio_status": "not_started",
                "frame_cut_record_status": "matched",
                "content_update_record_status": (
                    "matched" if CONTENT_UPDATE_FRAME_THINNING_ENABLED else "not_required"
                ),
            })
            return result
        if (
            previous_frame_cut_record is None
            and not frame_cut_has_explicit_config
            and not CONTENT_UPDATE_FRAME_THINNING_ENABLED
        ):
            result.update({
                "status": "skipped_existing_preprocess_file",
                "audio_status": "not_started",
                "frame_cut_record_status": "assumed_default",
                "content_update_record_status": "not_required",
            })
            return result

        delete_movie_preprocess_files(existing_preprocess_files)
        result.update({
            "frame_cut_record_status": "matched" if frame_cut_matches else "missing_or_changed",
            "content_update_record_status": "matched" if content_update_matches else "missing_or_changed",
            "reprocess_reason": "frame_cut_or_content_update_changed_or_unrecorded",
            "deleted_existing_preprocess_files": ";".join(str(path) for path in existing_preprocess_files),
        })

    if remaining_after_initial_frame_cut <= 0:
        result.update({
            "status": "skipped_no_frames_after_frame_cut",
            "written_frame_pairs": 0,
            "audio_status": "skipped_zero_duration",
        })
        return result

    sampled_frames = iter_sampled_source_frames(
        video_path=video_path,
        source_frame_indices=frame_cut_source_frame_indices,
        effective_fps=processing_fps,
    )
    result["sampled_frame_count"] = int(len(sampled_frames))
    if not sampled_frames:
        result.update({
            "status": "skipped_no_sampled_frames",
            "audio_status": "not_started",
        })
        return result

    _, crop_info = preprocess_stacked_frames_remove_letterbox(sampled_frames)
    trim_info = estimate_source_frame_indices_uniform_trim(
        video_path=video_path,
        crop_info=crop_info,
        source_frame_indices=frame_cut_source_frame_indices,
    )
    result.update({
        "crop_left": int(crop_info["crop_left"]),
        "crop_right": int(crop_info["crop_right"]),
        "cropped_width": int(crop_info["cropped_width"]),
        "removed_left_px": int(crop_info["removed_left_px"]),
        "removed_right_px": int(crop_info["removed_right_px"]),
    })
    result.update(trim_info)

    start_index = frame_cut_start_index + trim_info["removed_from_start"]
    end_index = frame_cut_end_index - trim_info["removed_from_end"]
    trim_source_frame_indices = source_frame_indices[start_index:max(start_index, end_index)]
    remaining_after_trim = len(trim_source_frame_indices)
    result.update({
        "trim_start_frame_index": int(start_index),
        "trim_end_frame_index": int(max(start_index, end_index)),
        "trim_start_source_frame_index": int(trim_source_frame_indices[0]) if trim_source_frame_indices else -1,
        "trim_end_source_frame_index": int(trim_source_frame_indices[-1] + 1) if trim_source_frame_indices else -1,
        "remaining_frame_pairs_after_trim": int(remaining_after_trim),
    })

    if remaining_after_trim <= 0:
        result.update({
            "status": "skipped_no_frames_after_trim",
            "written_frame_pairs": 0,
            "audio_status": "skipped_zero_duration",
        })
        return result

    source_trim_duration_sec = float(remaining_after_trim / processing_fps) if processing_fps > 0 else 0.0
    result["source_trim_duration_sec"] = source_trim_duration_sec
    output_fps = float(processing_fps)

    content_update_indices_by_view, content_update_info = select_content_update_source_frame_indices_by_view(
        video_path=video_path,
        source_frame_indices=trim_source_frame_indices,
        crop_info=crop_info,
        effective_fps=output_fps,
        source_duration_sec=source_trim_duration_sec,
        diff_threshold=content_update_duplicate_diff_threshold,
    )
    result.update(content_update_info)
    front_kept_frame_indices = content_update_indices_by_view.get("front", [])
    rear_kept_frame_indices = content_update_indices_by_view.get("rear", [])
    front_kept_frame_index_set = set(front_kept_frame_indices)
    rear_kept_frame_index_set = set(rear_kept_frame_indices)
    front_output_fps = float(content_update_info.get("front_content_update_output_fps", output_fps))
    rear_output_fps = float(content_update_info.get("rear_content_update_output_fps", output_fps))
    output_fps = front_output_fps if np.isclose(front_output_fps, rear_output_fps) else np.nan
    result["output_fps"] = float(output_fps) if np.isfinite(output_fps) else np.nan

    if not front_kept_frame_indices and not rear_kept_frame_indices:
        result.update({
            "status": "skipped_no_frames_after_content_update_thinning",
            "written_frame_pairs": 0,
            "written_front_frame_pairs": 0,
            "written_rear_frame_pairs": 0,
            "audio_status": "skipped_zero_duration",
        })
        return result

    last_kept_frame_index = max(front_kept_frame_indices + rear_kept_frame_indices)

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Failed to open video: {video_path}")

    front_writer = None
    rear_writer = None
    frame_pair_index = 0
    written_front_frame_pairs = 0
    written_rear_frame_pairs = 0

    try:
        while capture.isOpened():
            success, stacked_frame_bgr = capture.read()
            if not success:
                break
            if frame_pair_index > last_kept_frame_index:
                break

            should_write_front = frame_pair_index in front_kept_frame_index_set
            should_write_rear = frame_pair_index in rear_kept_frame_index_set
            if should_write_front or should_write_rear:
                cropped_frame_bgr = apply_side_crop(
                    stacked_frame_bgr,
                    crop_info["crop_left"],
                    crop_info["crop_right"],
                )
                front_frame_bgr, rear_frame_bgr = split_front_rear_frame(cropped_frame_bgr)

                if should_write_front and front_writer is None:
                    front_writer = create_mp4_writer(front_output_path, front_output_fps, front_frame_bgr.shape)
                if should_write_rear and rear_writer is None:
                    rear_writer = create_mp4_writer(rear_output_path, rear_output_fps, rear_frame_bgr.shape)
                if (should_write_front and written_front_frame_pairs == 0) or (should_write_rear and written_rear_frame_pairs == 0):
                    result.update({
                        "output_width": int(front_frame_bgr.shape[1]),
                        "output_height": int(front_frame_bgr.shape[0]),
                    })

                if should_write_front and front_writer is not None:
                    front_writer.write(front_frame_bgr)
                    written_front_frame_pairs += 1
                if should_write_rear and rear_writer is not None:
                    rear_writer.write(rear_frame_bgr)
                    written_rear_frame_pairs += 1

            frame_pair_index += 1
    finally:
        capture.release()
        if front_writer is not None:
            front_writer.release()
        if rear_writer is not None:
            rear_writer.release()

    trim_start_sec = float(start_index / processing_fps) if processing_fps > 0 else 0.0
    trim_duration_sec = source_trim_duration_sec
    audio_status = extract_trimmed_audio_to_wav(
        video_path=video_path,
        output_wav_path=audio_output_path,
        start_sec=trim_start_sec,
        duration_sec=trim_duration_sec,
        overwrite=overwrite,
    )

    result.update({
        "status": "written",
        "written_frame_pairs": int(min(written_front_frame_pairs, written_rear_frame_pairs)),
        "written_front_frame_pairs": int(written_front_frame_pairs),
        "written_rear_frame_pairs": int(written_rear_frame_pairs),
        "trim_start_sec": trim_start_sec,
        "trim_duration_sec": trim_duration_sec,
        "audio_status": audio_status,
    })
    return result


# 関数メモ: run_movie_preprocess_for_dataset
# - 対象動画リスト全体に前処理を適用し、manifest を保存します。
# - 上書き設定や stale 削除を含め、データセット単位の再現可能な前処理を実行します。

def run_movie_preprocess_for_dataset(
    output_base_dir: Path = MOVIE_PREPROCESS_OUTPUT_DIR,
    manifest_path: Path = MOVIE_PREPROCESS_MANIFEST_PATH,
    overwrite: bool = MOVIE_PREPROCESS_OVERWRITE,
) -> pd.DataFrame:
    all_targets = find_movie_preprocess_targets()
    targets, requested_target_video_names = filter_movie_preprocess_targets_by_video_name(all_targets)
    frame_cut_config = load_frame_cut_config()
    all_frame_cut = frame_cut_config.get("all", {"front": 0, "rear": 0})
    individual_frame_cut_count = len([name for name in frame_cut_config if name != "all"])
    individual_duplicate_threshold_count = len([
        name for name, config in frame_cut_config.items()
        if name != "all" and config.get("duplicate_diff_threshold") is not None
    ])
    previous_manifest_df = load_movie_preprocess_manifest(manifest_path)
    previous_frame_cut_records = build_previous_frame_cut_records(previous_manifest_df)
    print("movie preprocess total source target count:", len(all_targets))
    print("movie preprocess selected target count:", len(targets))
    if requested_target_video_names:
        print("movie preprocess selected video names:", sorted(requested_target_video_names))
    print("frame cut all front/rear:", all_frame_cut.get("front", 0), all_frame_cut.get("rear", 0))
    print("frame cut individual target count:", individual_frame_cut_count)
    print("frame cut individual duplicate threshold count:", individual_duplicate_threshold_count)
    print("previous frame cut record count:", len(previous_frame_cut_records))
    cleanup_stale_movie_preprocess_outputs(
        targets=all_targets,
        output_base_dir=output_base_dir,
        delete_stale=MOVIE_PREPROCESS_DELETE_STALE,
    )
    rows: list[dict[str, Any]] = []

    for target_index, target in enumerate(targets, start=1):
        category = target["category"]
        environment = target["environment"]
        video_path = target["video_path"]
        print(f"[{target_index}/{len(targets)}] {category}/{environment}: {video_path.name}")
        try:
            result = preprocess_movie_to_front_rear_audio(
                video_path=video_path,
                category=category,
                environment=environment,
                output_base_dir=output_base_dir,
                overwrite=overwrite,
                frame_cut_config=frame_cut_config,
                previous_frame_cut_records=previous_frame_cut_records,
            )
        except Exception as error:
            front_output_path, rear_output_path, audio_output_path = build_movie_preprocess_output_paths(
                video_path=video_path,
                category=category,
                environment=environment,
                output_base_dir=output_base_dir,
            )
            result = {
                "category": category,
                "environment": environment,
                "input_video_path": str(video_path),
                "front_output_path": str(front_output_path),
                "rear_output_path": str(rear_output_path),
                "audio_output_path": str(audio_output_path),
                "status": "error",
                "audio_status": "error",
                "error": repr(error),
            }

        rows.append(result)
        print(
            "  status:",
            result.get("status"),
            "written_frame_pairs:",
            result.get("written_frame_pairs"),
            "written_front/rear:",
            result.get("written_front_frame_pairs"),
            result.get("written_rear_frame_pairs"),
            "content_update_removed_front/rear:",
            result.get("front_content_update_removed_frame_pairs"),
            result.get("rear_content_update_removed_frame_pairs"),
            "front/rear_output_fps:",
            result.get("front_content_update_output_fps"),
            result.get("rear_content_update_output_fps"),
            "audio_status:",
            result.get("audio_status"),
        )

    run_manifest_df = pd.DataFrame(rows)
    if requested_target_video_names and not previous_manifest_df.empty:
        processed_keys = {movie_preprocess_manifest_row_key(row) for row in rows}
        keep_previous_mask = previous_manifest_df.apply(
            lambda row: movie_preprocess_manifest_row_key(row) not in processed_keys,
            axis=1,
        )
        manifest_df = pd.concat(
            [previous_manifest_df.loc[keep_previous_mask], run_manifest_df],
            ignore_index=True,
            sort=False,
        )
    else:
        manifest_df = run_manifest_df
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_df.to_csv(manifest_path, index=False)
    print("movie preprocess manifest saved to:", manifest_path)
    if not manifest_df.empty and "status" in manifest_df:
        print("status counts:")
        print(manifest_df["status"].value_counts(dropna=False).to_dict())
    if not manifest_df.empty and "audio_status" in manifest_df:
        print("audio status counts:")
        print(manifest_df["audio_status"].value_counts(dropna=False).to_dict())
    return manifest_df


if VIDEO_PROCESSING_ENABLED:
    movie_preprocess_manifest_df = run_movie_preprocess_for_dataset()
    display(movie_preprocess_manifest_df.head())
else:
    print("Skip movie preprocessing because VIDEO_PROCESSING_ENABLED is False.")


movie preprocess total source target count: 193
movie preprocess selected target count: 1
movie preprocess selected video names: ['1041']
frame cut all front/rear: 0 20
frame cut individual target count: 6
frame cut individual duplicate threshold count: 1
previous frame cut record count: 193
stale movie_preprocess artifacts: 0
[1/1] normal/indoor: 1041.mp4
  status: written written_frame_pairs: 145 written_front/rear: 145 145 content_update_removed_front/rear: 289 289 front/rear_output_fps: 10.023041474654377 10.023041474654377 audio_status: written
movie preprocess manifest saved to: ../data/movie_preprocess/movie_preprocess_manifest.csv
status counts:
{'written': 193}
audio status counts:
{'written': 193}


,category,environment,input_video_path,front_output_path,rear_output_path,audio_output_path,input_frame_count,input_fps,input_width,input_height,...,content_update_tile_top_fraction,content_update_tile_active_threshold_ratio,content_update_tile_min_active_tiles,content_update_tile_min_active_ratio,previous_content_update_tile_grid_rows,previous_content_update_tile_grid_cols,previous_content_update_tile_top_fraction,previous_content_update_tile_active_threshold_ratio,previous_content_update_tile_min_active_tiles,previous_content_update_tile_min_active_ratio
0,normal,outdoor,../data/ori/normal/1002.mp4,../data/movie_preprocess/normal/outdoor/1002_f...,../data/movie_preprocess/normal/outdoor/1002_r...,../data/movie_preprocess/normal/outdoor/1002_a...,469,30.0,1920,1080,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,normal,outdoor,../data/ori/normal/1003.mp4,../data/movie_preprocess/normal/outdoor/1003_f...,../data/movie_preprocess/normal/outdoor/1003_r...,../data/movie_preprocess/normal/outdoor/1003_a...,463,30.0,1920,1080,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,normal,outdoor,../data/ori/normal/1004.mp4,../data/movie_preprocess/normal/outdoor/1004_f...,../data/movie_preprocess/normal/outdoor/1004_r...,../data/movie_preprocess/normal/outdoor/1004_a...,487,30.0,1920,1080,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,normal,outdoor,../data/ori/normal/1005.mp4,../data/movie_preprocess/normal/outdoor/1005_f...,../data/movie_preprocess/normal/outdoor/1005_r...,../data/movie_preprocess/normal/outdoor/1005_a...,480,30.0,1920,1080,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,normal,outdoor,../data/ori/normal/1006.mp4,../data/movie_preprocess/normal/outdoor/1006_f...,../data/movie_preprocess/normal/outdoor/1006_r...,../data/movie_preprocess/normal/outdoor/1006_a...,465,30.0,1920,1080,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
